# tensor-wraps-ndarray — ex2: wrap ndarray with dtype-aware routing policy

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-wraps-ndarray`. Running the final beacon cell reports progress against the `PyTorch: tensor from ndarray` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: tensor from ndarray` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-wraps-ndarray`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-wraps-ndarray"
DD_SUBTOPIC = "PyTorch: tensor from ndarray"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `from_numpy` / `as_tensor` / `tensor` — quick refresher

Three ways to wrap a numpy array, with very different aliasing semantics:

- `t.from_numpy(arr)` — **always shares memory** with `arr`. Same dtype as arr. Raises if arr dtype is unsupported (e.g. uint16). Mutating one mutates the other.
- `t.as_tensor(arr)` — **shares memory IFF dtype/device match the target**; otherwise copies. Effectively `from_numpy` when types align, `tensor` when they don't.
- `t.tensor(arr)` — **always copies**. Independent storage. Mutating one does NOT affect the other.

**Why a policy matters.** ARENA pipelines load int32 numpy arrays from disk but train with int64 indices and float32 features. A policy that picks the right wrapper per dtype avoids the "silently aliased buffer" footgun while still saving a copy when the dtypes match.

### Exercise 2 — wrap ndarray with dtype-aware routing policy

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `t.from_numpy` / `t.as_tensor` / `t.tensor` selection based on input ndarray dtype + a target dtype, producing the correct aliasing-or-copy outcome per case.
> Keywords: from_numpy, as_tensor, tensor, dtype-routing, aliasing
> ```

**KCs targeted:** `from-numpy-shares-storage`, `as-tensor-conditional-copy`

Implement `ex2_wrap_ndarray(arr, target_dtype)`. A small routing policy that picks the right wrapper:

- `arr` is a numpy ndarray.
- `target_dtype` is a torch dtype (e.g. `t.float32`, `t.int64`).

Rules:
1. If the numpy dtype already corresponds to `target_dtype` (e.g. arr is `float32` and target is `t.float32`), use `t.from_numpy(arr)` — fast, **aliased**, no copy.
2. Otherwise, use `t.tensor(arr, dtype=target_dtype)` — explicit copy + cast. The result must NOT alias `arr`.

**Use `t.from_numpy(arr).dtype` to check the would-be dtype** of a numpy array without doing the wrap yet (or check `arr.dtype.name` against torch dtype names — both work).

Return a dict: `{'tensor': out_tensor, 'aliased': bool}` where `aliased` is True iff the returned tensor shares memory with `arr` (i.e. you went down branch 1).

The test will verify aliasing the hard way: mutate `arr` in-place and check whether `out_tensor` reflects the change.

In [ ]:
def ex2_wrap_ndarray(arr: np.ndarray, target_dtype: t.dtype) -> dict:
    """Route to from_numpy (alias) or tensor (copy+cast) per dtype match."""
    raise NotImplementedError()


def _test_ex2():
    # Case 1 — float32 ndarray, target float32 → from_numpy → aliased.
    arr1 = np.array([1.0, 2.0, 3.0], dtype=np.float32)
    out1 = ex2_wrap_ndarray(arr1, t.float32)
    assert set(out1.keys()) == {'tensor', 'aliased'}
    ten1 = out1['tensor']
    assert ten1.dtype == t.float32, f'expected float32, got {ten1.dtype}'
    assert out1['aliased'] is True, 'matching dtype should alias (from_numpy)'
    # Mutate source — tensor should reflect it.
    arr1[0] = 99.0
    assert ten1[0].item() == 99.0, 'aliased tensor must reflect ndarray mutation'

    # Case 2 — int32 ndarray, target int64 → tensor → copy.
    arr2 = np.array([1, 2, 3], dtype=np.int32)
    out2 = ex2_wrap_ndarray(arr2, t.int64)
    ten2 = out2['tensor']
    assert ten2.dtype == t.int64, f'expected int64, got {ten2.dtype}'
    assert out2['aliased'] is False, 'dtype mismatch should COPY, not alias'
    arr2[0] = 999
    assert ten2[0].item() == 1, 'copied tensor must NOT reflect ndarray mutation'

    # Case 3 — float64 ndarray, target float64 → from_numpy → aliased.
    arr3 = np.array([0.1, 0.2], dtype=np.float64)
    out3 = ex2_wrap_ndarray(arr3, t.float64)
    ten3 = out3['tensor']
    assert ten3.dtype == t.float64
    assert out3['aliased'] is True
    arr3[1] = -1.5
    assert ten3[1].item() == -1.5

    # Case 4 — float64 ndarray, target float32 → tensor → copy + downcast.
    arr4 = np.array([1.5, 2.5], dtype=np.float64)
    out4 = ex2_wrap_ndarray(arr4, t.float32)
    ten4 = out4['tensor']
    assert ten4.dtype == t.float32, f'expected float32 after downcast, got {ten4.dtype}'
    assert out4['aliased'] is False
    arr4[0] = 100.0
    assert abs(ten4[0].item() - 1.5) < 1e-6, 'downcast result should NOT alias'

    # Case 5 — int64 ndarray, target int64 → from_numpy → aliased.
    arr5 = np.array([10, 20, 30], dtype=np.int64)
    out5 = ex2_wrap_ndarray(arr5, t.int64)
    assert out5['aliased'] is True
    arr5[2] = 777
    assert out5['tensor'][2].item() == 777
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_wrap_ndarray(arr: np.ndarray, target_dtype: t.dtype) -> dict:
    natural = t.from_numpy(arr).dtype
    if natural == target_dtype:
        return {'tensor': t.from_numpy(arr), 'aliased': True}
    return {'tensor': t.tensor(arr, dtype=target_dtype), 'aliased': False}
```

**Why check via `t.from_numpy(arr).dtype` not `arr.dtype.name`.** Torch and numpy don't always share dtype spellings (`np.int64` vs `t.int64` happen to align, but `np.bool_` vs `t.bool` don't on all builds). Routing through `from_numpy` makes torch tell you the canonical mapping. The check is cheap because `from_numpy` just inspects metadata — no data copy at that moment.

**Why not `t.as_tensor`?** `as_tensor` already does the routing we built — same dtype → share, different dtype → copy. We built the policy ourselves to drive home what `as_tensor` is actually doing under the hood (Apply-level mastery), and to return the explicit `'aliased'` flag callers want.

**The aliasing hazard.** Aliased tensors are dangerous if you later `.to('cuda')` (which copies, breaking alias) or pass to a module that calls `.contiguous()` (also a copy). For training data loaded once at startup, alias is fine; for online data streams, prefer the explicit copy.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()